# Visualization Demo

This notebook demonstrates the final subset-visualization API in `pygeoinf`.

Examples covered:
- `Subset.plot()` in 2D
- `plot_slice()` for a 2D slice through a 3D set
- full 3D voxel rendering
- exact polyhedral slicing with vertex payloads

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from pygeoinf.hilbert_space import EuclideanSpace
from pygeoinf.plot import plot_slice
from pygeoinf.subsets import Ball, HalfSpace, PolyhedralSet
from pygeoinf.subspaces import AffineSubspace


def show_and_close(fig):
    display(fig)
    plt.close(fig)


warnings.filterwarnings(
    "ignore",
    message="Constructing an AffineSubspace without a solver",
    category=UserWarning,
)

np.set_printoptions(precision=3, suppress=True)

## 1. Plot a 2D Ball with `Subset.plot()`

For a 2D Euclidean domain, `Subset.plot()` can build the default affine subspace automatically.

In [ ]:
domain2 = EuclideanSpace(2)
ball2 = Ball(domain2, center=domain2.zero, radius=1.0)

fig2, ax2, mask2 = ball2.plot(bounds=(-1.4, 1.4, -1.4, 1.4), grid_size=120, show_plot=False)
print("2D payload shape:", mask2.shape)
show_and_close(fig2)

## 2. Slice a 3D Ball Along a 2D Affine Plane

For higher-dimensional sets, define the slice subspace explicitly and call `plot_slice()`.

In [ ]:
domain3 = EuclideanSpace(3)
ball3 = Ball(domain3, center=domain3.zero, radius=1.0)

slice_plane = AffineSubspace.from_tangent_basis(
    domain3,
    [domain3.basis_vector(0), domain3.basis_vector(1)],
    translation=0.2 * domain3.basis_vector(2),
)

fig_slice, ax_slice, mask_slice = plot_slice(
    ball3,
    slice_plane,
    bounds=(-1.2, 1.2, -1.2, 1.2),
    grid_size=100,
    show_plot=False,
)
print("2D slice payload shape:", mask_slice.shape)
show_and_close(fig_slice)

## 3. Render a Full 3D Ball

A 3D affine subspace yields the voxel-based rendering path for non-polyhedral sets.

In [ ]:
full_space = AffineSubspace.from_tangent_basis(
    domain3,
    [domain3.basis_vector(0), domain3.basis_vector(1), domain3.basis_vector(2)],
)

fig3d, ax3d, mask3d = plot_slice(
    ball3,
    full_space,
    bounds=(-1.1, 1.1, -1.1, 1.1, -1.1, 1.1),
    grid_size=22,
    show_plot=False,
)
print("3D voxel payload shape:", mask3d.shape)
show_and_close(fig3d)

## 4. Exact Polyhedral Slice

`PolyhedralSet` uses the exact half-space intersection path, so the payload is geometric vertices instead of a boolean mask.

In [ ]:
cube_halfspaces = [
    HalfSpace(domain3, domain3.basis_vector(0), 0.8),
    HalfSpace(domain3, -domain3.basis_vector(0), 0.8),
    HalfSpace(domain3, domain3.basis_vector(1), 0.8),
    HalfSpace(domain3, -domain3.basis_vector(1), 0.8),
    HalfSpace(domain3, domain3.basis_vector(2), 0.8),
    HalfSpace(domain3, -domain3.basis_vector(2), 0.8),
]
cube = PolyhedralSet(domain3, cube_halfspaces)

diagonal_plane = AffineSubspace.from_tangent_basis(
    domain3,
    [
        domain3.basis_vector(0) + domain3.basis_vector(1),
        domain3.basis_vector(2),
    ],
)

fig_poly, ax_poly, verts_poly = plot_slice(
    cube,
    diagonal_plane,
    bounds=(-1.2, 1.2, -1.2, 1.2),
    show_plot=False,
)
print("Exact polygon vertices:\n", verts_poly)
show_and_close(fig_poly)

## 5. Quick Sanity Checks

These checks keep the demo honest about the payload types returned by the visualization layer.

In [ ]:
assert mask2.ndim == 2
assert mask_slice.ndim == 2
assert mask3d.ndim == 3
assert verts_poly.ndim == 2 and verts_poly.shape[1] == 2

print("Visualization demo checks passed.")